In [ ]:
# ============================================================================
# VISUALIZATION: 3D t-SNE Class Separation Map (BUY vs SELL)
# ============================================================================
print("\n" + "="*70)
print("Creating 3D t-SNE Class Separation Visualization (Buy vs Sell)...")
print("="*70)

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================================
# STEP 1: Collect data from all sliding windows OR choose one window
# ============================================================================

# OPTION 1: Find common features across all windows
print("Finding common features across all windows...")
common_features = set(selected_features_per_window[f'Window_{first_slide+1}'])
for i in range(first_slide, second_slide):
    common_features = common_features.intersection(set(selected_features_per_window[f'Window_{i+1}']))

common_features = sorted(list(common_features))
print(f"Common features across all windows: {len(common_features)}")

if len(common_features) < 10:
    print(f"⚠️ Warning: Only {len(common_features)} common features found.")
    print("Switching to SINGLE WINDOW visualization (best performing window)...")
    
    # Choose the best performing window
    best_window_name = max(window_performance_metrics.keys(), 
                          key=lambda k: window_performance_metrics[k]['test_accuracy'])
    best_window_idx = int(best_window_name.split('_')[1]) - 1 + first_slide
    
    print(f"Selected: {best_window_name} (Test Accuracy: {window_performance_metrics[best_window_name]['test_accuracy']:.4f})")
    
    # Get data for best window only
    a = nn * best_window_idx
    selected_features_best = selected_features_per_window[best_window_name]
    X_sample = np.array(df[selected_features_best][n+a+1:n+a+1+nn])
    y_sample = df['LABEL'][n+a+1:n+a+1+nn].values
    window_sample = [best_window_name] * len(X_sample)
    
else:
    # Use common features across all windows
    print(f"✅ Using {len(common_features)} common features")
    
    X_all_windows = []
    y_all_windows = []
    window_labels = []
    
    for i in range(first_slide, second_slide):
        a = nn * i
        X_window = np.array(df[common_features][:n+a+1])
        y_window = df['LABEL'][:n+a+1].values
        
        X_all_windows.append(X_window)
        y_all_windows.append(y_window)
        window_labels.extend([f'Window_{i+1}'] * len(X_window))
    
    X_combined = np.vstack(X_all_windows)
    y_combined = np.hstack(y_all_windows)
    
    # Sample for computational efficiency
    n_samples_to_plot = 5000
    if len(X_combined) > n_samples_to_plot:
        sample_indices = np.random.choice(len(X_combined), n_samples_to_plot, replace=False)
        X_sample = X_combined[sample_indices]
        y_sample = y_combined[sample_indices]
        window_sample = [window_labels[idx] for idx in sample_indices]
    else:
        X_sample = X_combined
        y_sample = y_combined
        window_sample = window_labels

# Sample for computational efficiency
n_samples_to_plot = 5000
if len(X_sample) > n_samples_to_plot:
    sample_indices = np.random.choice(len(X_sample), n_samples_to_plot, replace=False)
    X_sample_final = X_sample[sample_indices]
    y_sample_final = y_sample[sample_indices]
    window_sample_final = [window_sample[idx] for idx in sample_indices]
else:
    X_sample_final = X_sample
    y_sample_final = y_sample
    window_sample_final = window_sample

print(f"Sample size for t-SNE: {len(X_sample_final)} samples")
print(f"Buy samples: {np.sum(y_sample_final == 1)}")
print(f"Sell samples: {np.sum(y_sample_final == 0)}")

# ============================================================================
# STEP 2: Apply 3D t-SNE
# ============================================================================

print("Applying 3D t-SNE (this may take a few minutes)...")
tsne_3d = TSNE(n_components=3, random_state=42, 
               perplexity=min(50, len(X_sample_final)//3),
               n_iter=1000, verbose=1)
tsne_results_3d = tsne_3d.fit_transform(X_sample_final)

print("t-SNE completed!")

# ============================================================================
# VISUALIZATION 1: 3D Scatter Plots (Buy vs Sell)
# ============================================================================

plt.rcParams.update({'font.size': 14})
fig = plt.figure(figsize=(24, 8))

# Define colors
class_colors = {0: '#FF3333', 1: '#33FF33'}  # 0=Sell (Red), 1=Buy (Green)
class_names = {0: 'SELL', 1: 'BUY'}

# Plot 1: Colored by class (Buy vs Sell)
ax1 = fig.add_subplot(131, projection='3d')

for class_label in [0, 1]:
    mask = y_sample_final == class_label
    ax1.scatter(tsne_results_3d[mask, 0], 
               tsne_results_3d[mask, 1], 
               tsne_results_3d[mask, 2],
               c=class_colors[class_label], 
               label=class_names[class_label],
               s=80, alpha=0.6, edgecolors='black', linewidth=0.5)

ax1.set_xlabel('t-SNE Component 1', fontsize=20, fontweight='bold', labelpad=20)
ax1.set_ylabel('t-SNE Component 2', fontsize=20, fontweight='bold', labelpad=20)
ax1.set_zlabel('t-SNE Component 3', fontsize=20, fontweight='bold', labelpad=20)
#ax1.set_title('3D Class Separation: BUY vs SELL\n(XRP/USDT 1-Min Trading)',
            #fontsize=40, pad=20, fontweight='bold')
ax1.legend(fontsize=25, loc='upper right', framealpha=0.9, markerscale=2)
ax1.view_init(elev=20, azim=45)
ax1.grid(True, alpha=0.3)

# Plot 2: Same view from different angle
ax2 = fig.add_subplot(132, projection='3d')

for class_label in [0, 1]:
    mask = y_sample_final == class_label
    ax2.scatter(tsne_results_3d[mask, 0], 
               tsne_results_3d[mask, 1], 
               tsne_results_3d[mask, 2],
               c=class_colors[class_label], 
               label=class_names[class_label],
               s=80, alpha=0.6, edgecolors='black', linewidth=0.5)

ax2.set_xlabel('t-SNE Component 1', fontsize=20, fontweight='bold', labelpad=20)
ax2.set_ylabel('t-SNE Component 2', fontsize=20, fontweight='bold', labelpad=20)
ax2.set_zlabel('t-SNE Component 3', fontsize=20, fontweight='bold', labelpad=20)
ax2.set_title('3D Class Separation: BUY vs SELL\n(Different Viewing Angle)',
             fontsize=40, pad=20, fontweight='bold')
ax2.legend(fontsize=25, loc='upper right', framealpha=0.9, markerscale=2)
ax2.view_init(elev=30, azim=120)  # Different angle
ax2.grid(True, alpha=0.3)

# Plot 3: Density-based coloring (how concentrated each class is)
ax3 = fig.add_subplot(133, projection='3d')

# Calculate distance to class centroid for each point
buy_mask = y_sample_final == 1
sell_mask = y_sample_final == 0

buy_centroid = np.mean(tsne_results_3d[buy_mask], axis=0)
sell_centroid = np.mean(tsne_results_3d[sell_mask], axis=0)

buy_distances = np.linalg.norm(tsne_results_3d[buy_mask] - buy_centroid, axis=1)
sell_distances = np.linalg.norm(tsne_results_3d[sell_mask] - sell_centroid, axis=1)

scatter_buy = ax3.scatter(tsne_results_3d[buy_mask, 0], 
                         tsne_results_3d[buy_mask, 1], 
                         tsne_results_3d[buy_mask, 2],
                         c=buy_distances, cmap='Greens_r', 
                         s=80, alpha=0.6, edgecolors='black', linewidth=0.5,
                         label='BUY')

scatter_sell = ax3.scatter(tsne_results_3d[sell_mask, 0], 
                          tsne_results_3d[sell_mask, 1], 
                          tsne_results_3d[sell_mask, 2],
                          c=sell_distances, cmap='Reds_r', 
                          s=80, alpha=0.6, edgecolors='black', linewidth=0.5,
                          label='SELL')

ax3.set_xlabel('t-SNE Component 1', fontsize=20, fontweight='bold', labelpad=20)
ax3.set_ylabel('t-SNE Component 2', fontsize=20, fontweight='bold', labelpad=20)
ax3.set_zlabel('t-SNE Component 3', fontsize=20, fontweight='bold', labelpad=20)
#ax3.set_title('Class Density Distribution\n(Darker = Closer to Centroid)',
             #fontsize=40, pad=20, fontweight='bold')
ax3.legend(fontsize=25, loc='upper right', framealpha=0.9, markerscale=2)
ax3.view_init(elev=20, azim=45)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/class_separation_3d_buy_vs_sell.tiff', bbox_inches='tight', dpi=300)
plt.savefig('/kaggle/working/class_separation_3d_buy_vs_sell.svg', bbox_inches='tight', dpi=100)
plt.show()

# ============================================================================
# VISUALIZATION 2: 2D Projections for Better Clarity
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(27, 8))

projections = [
    (0, 1, 't-SNE Component 1', 't-SNE Component 2'),
    (0, 2, 't-SNE Component 1', 't-SNE Component 3'),
    (1, 2, 't-SNE Component 2', 't-SNE Component 3')
]

for idx, (dim1, dim2, xlabel, ylabel) in enumerate(projections):
    ax = axes[idx]
    
    for class_label in [0, 1]:
        mask = y_sample_final == class_label
        ax.scatter(tsne_results_3d[mask, dim1], 
                  tsne_results_3d[mask, dim2],
                  c=class_colors[class_label], 
                  label=class_names[class_label],
                  s=50, alpha=0.6, edgecolors='black', linewidth=0.3)
    
    ax.set_xlabel(xlabel, fontsize=30, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=30, fontweight='bold')
    #ax.set_title(f'2D Projection\n{xlabel} vs {ylabel}',
                #fontsize=18, fontweight='bold', pad=15)
    ax.legend(fontsize=25, framealpha=0.9, markerscale=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/class_separation_2d_projections.tiff', bbox_inches='tight', dpi=300)
plt.savefig('/kaggle/working/class_separation_2d_projections.svg', bbox_inches='tight', dpi=100)
plt.show()

# ============================================================================
# VISUALIZATION 3: Class Overlap Analysis
# ============================================================================

from scipy.spatial.distance import cdist
from sklearn.neighbors import NearestNeighbors

print("\n" + "="*70)
print("CLASS SEPARATION METRICS")
print("="*70)

# Calculate centroids
buy_centroid = np.mean(tsne_results_3d[buy_mask], axis=0)
sell_centroid = np.mean(tsne_results_3d[sell_mask], axis=0)

centroid_distance = np.linalg.norm(buy_centroid - sell_centroid)

print(f"\n📊 Class Distribution:")
print(f"  - BUY samples: {np.sum(buy_mask)} ({100*np.sum(buy_mask)/len(y_sample_final):.1f}%)")
print(f"  - SELL samples: {np.sum(sell_mask)} ({100*np.sum(sell_mask)/len(y_sample_final):.1f}%)")

print(f"\n📏 Centroid Separation:")
print(f"  - BUY centroid: {buy_centroid}")
print(f"  - SELL centroid: {sell_centroid}")
print(f"  - Distance between centroids: {centroid_distance:.4f}")

# Intra-class variance (compactness)
buy_variance = np.mean(np.var(tsne_results_3d[buy_mask], axis=0))
sell_variance = np.mean(np.var(tsne_results_3d[sell_mask], axis=0))

print(f"\n📐 Intra-Class Variance (Compactness):")
print(f"  - BUY class variance: {buy_variance:.4f}")
print(f"  - SELL class variance: {sell_variance:.4f}")

# Calculate overlap using nearest neighbors
nn_model = NearestNeighbors(n_neighbors=10)
nn_model.fit(tsne_results_3d)

buy_nn_distances, buy_nn_indices = nn_model.kneighbors(tsne_results_3d[buy_mask])
buy_neighbor_classes = y_sample_final[buy_nn_indices]
buy_purity = np.mean(buy_neighbor_classes == 1)

sell_nn_distances, sell_nn_indices = nn_model.kneighbors(tsne_results_3d[sell_mask])
sell_neighbor_classes = y_sample_final[sell_nn_indices]
sell_purity = np.mean(sell_neighbor_classes == 0)

print(f"\n🎯 Neighborhood Purity (10-NN):")
print(f"  - BUY class purity: {buy_purity:.2%}")
print(f"  - SELL class purity: {sell_purity:.2%}")
print(f"  - Overall separation quality: {(buy_purity + sell_purity)/2:.2%}")

# Silhouette score
from sklearn.metrics import silhouette_score
silhouette = silhouette_score(tsne_results_3d, y_sample_final)
print(f"\n✨ Silhouette Score: {silhouette:.4f}")
print(f"   (Range: -1 to 1, higher is better. >0.5 = good separation)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

# Save t-SNE coordinates with labels
tsne_df = pd.DataFrame({
    'tSNE_1': tsne_results_3d[:, 0],
    'tSNE_2': tsne_results_3d[:, 1],
    'tSNE_3': tsne_results_3d[:, 2],
    'Class': y_sample_final,
    'Class_Name': ['SELL' if label == 0 else 'BUY' for label in y_sample_final],
    'Window': window_sample_final
})
tsne_df.to_csv('/kaggle/working/class_separation_tsne_coordinates.csv', index=False)

# Save separation metrics
metrics_dict = {
    'centroid_distance': centroid_distance,
    'buy_variance': buy_variance,
    'sell_variance': sell_variance,
    'buy_purity': buy_purity,
    'sell_purity': sell_purity,
    'silhouette_score': silhouette,
    'buy_count': int(np.sum(buy_mask)),
    'sell_count': int(np.sum(sell_mask))
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_df.to_csv('/kaggle/working/class_separation_metrics.csv', index=False)

print("\n✅ Class separation analysis complete!")
print("\nGenerated files:")
print("- class_separation_3d_buy_vs_sell.tiff/svg (3D scatter plots)")
print("- class_separation_2d_projections.tiff/svg (2D projections)")
print("- class_separation_tsne_coordinates.csv (all coordinates)")
print("- class_separation_metrics.csv (separation quality metrics)")

print("\n" + "="*70)

In [ ]:
from IPython.display import IFrame
IFrame('class_separation_3d_interactive.html', width=1000, height=800)

In [ ]:
# ============================================================================
# VISUALIZATION : Interactive 3D t-SNE Plot (Plotly)
# ============================================================================
import plotly.graph_objs as go
import plotly.io as pio

print("Creating interactive 3D Plotly visualization...")

# Prepare data for Plotly
buy_mask = y_sample_final == 1
sell_mask = y_sample_final == 0

trace_buy = go.Scatter3d(
    x=tsne_results_3d[buy_mask, 0],
    y=tsne_results_3d[buy_mask, 1],
    z=tsne_results_3d[buy_mask, 2],
    mode='markers',
    name='BUY',
    marker=dict(
        size=4,
        color='limegreen',
        opacity=0.75,
        line=dict(width=0.5, color='black')
    ),
    text=[f'Window: {w}<br>Class: BUY' for w, m in zip(window_sample_final, buy_mask) if m]
)

trace_sell = go.Scatter3d(
    x=tsne_results_3d[sell_mask, 0],
    y=tsne_results_3d[sell_mask, 1],
    z=tsne_results_3d[sell_mask, 2],
    mode='markers',
    name='SELL',
    marker=dict(
        size=4,
        color='red',
        opacity=0.75,
        line=dict(width=0.5, color='black')
    ),
    text=[f'Window: {w}<br>Class: SELL' for w, m in zip(window_sample_final, sell_mask) if m]
)

layout = go.Layout(
    title=dict(
        text='<b>Interactive 3D t-SNE Class Separation: BUY vs SELL</b>',
        x=0.5,
        font=dict(size=20)
    ),
    scene=dict(
        xaxis_title='t-SNE Component 1',
        yaxis_title='t-SNE Component 2',
        zaxis_title='t-SNE Component 3',
        xaxis=dict(backgroundcolor='rgb(240,240,240)', gridcolor='white', showbackground=True),
        yaxis=dict(backgroundcolor='rgb(240,240,240)', gridcolor='white', showbackground=True),
        zaxis=dict(backgroundcolor='rgb(240,240,240)', gridcolor='white', showbackground=True),
    ),
    legend=dict(
        x=0.8, y=0.9,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='black',
        borderwidth=1
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig_interactive = go.Figure(data=[trace_buy, trace_sell], layout=layout)

# Show in notebook (if supported)
fig_interactive.show()

# Save interactive version as HTML
pio.write_html(fig_interactive, '/kaggle/working/class_separation_3d_interactive.html', include_plotlyjs='cdn')

print("✅ Interactive 3D t-SNE visualization saved as 'class_separation_3d_interactive.html'")
